## ⚙️ Configuration

Set `BUSINESS_DATE_OVERRIDE` to a `"YYYY-MM-DD"` string to use a specific date, or leave it as `None` to pick up the date set in the platform's Execution Context.


In [ ]:
import requests
from datetime import date

# ── Override (set to a "YYYY-MM-DD" string to pin a specific date) ────────────
BUSINESS_DATE_OVERRIDE = None          # e.g. "2026-04-18"
# ─────────────────────────────────────────────────────────────────────────────

API_BASE = "http://localhost:8000/api/v1"

if BUSINESS_DATE_OVERRIDE:
    BUSINESS_DATE = BUSINESS_DATE_OVERRIDE
else:
    try:
        ctx = requests.get(f"{API_BASE}/etl/context", timeout=3).json()
        BUSINESS_DATE = ctx.get("business_date") or date.today().isoformat()
    except Exception:
        BUSINESS_DATE = date.today().isoformat()

NAMESPACE = BUSINESS_DATE.replace("-", "")   # compact YYYYMMDD for Spark DB names
print(f"Business date : {BUSINESS_DATE}")
print(f"Namespace     : {NAMESPACE}")


# Pipeline Orchestration

Trigger, monitor, and query pipelines via the platform API and Spark Connect.

In [ ]:
spark.sql(f"SHOW TABLES IN {NAMESPACE}").show(truncate=False)


In [ ]:
spark.sql(f"SELECT * FROM {NAMESPACE}.test_dw LIMIT 10").show(truncate=False)


In [ ]:
import sqlite3, pandas as pd, os

db_path = os.path.join(os.path.dirname(os.getcwd()), "data", "metadata.db")
con = sqlite3.connect(db_path)
df = pd.read_sql("SELECT id, name, conn_type, host, port, database, username, extra, created_at FROM connections ORDER BY name", con)
con.close()
df
